# RT-DETR + PvtV2-b2 on DIVA-HisDB CB55 (Colab)

Repeats the local PvtV2-b0 run with a **PvtV2-b2** backbone, everything else held
fixed: `imgsz=1152`, 100 epochs, batch 1 x grad_accum 2 (1000 optimizer steps),
constant LR 1e-4, EMA 0.993, seed 42, colour jitter, and the COCO-pretrained
encoder/decoder from `PekingU/rtdetr_r50vd`.

**Two arms**
1. `pvt_v2_b2_rand` — random backbone init (directly comparable to the local b0 row).
2. `pvt_v2_b2_in1k` — ImageNet-pretrained backbone. Tests whether backbone
   pretraining buys anything once encoder/decoder are already COCO-pretrained.

**PvtV2 stage indexing.** `PvtV2Config.stage_names` is `['stage1'..'stage4']` with no
`stem` entry, so the three stages at strides 8/16/32 are `out_indices=[1, 2, 3]`
(`[2, 3, 4]` raises `ValueError`). ConvNeXt enumerates `['stem','stage1'..'stage4']`
and would use `[2, 3, 4]` for the same stages.

**Resolution guard.** b2's stage-1 attention is much heavier than b0's. Section 6
measures the real peak and, if 1152 does not fit, drops to the largest multiple of 32
that does **and automatically adds b0 + stock R50-vd to the run list**, because the
existing local rows at 1152 would no longer be comparable.

Expected on Drive:

    MyDrive/Thesis/
      00_data/                                  <- CB55 subset, see section 4
      80_models/02_2stage/diva-hisdb/segmentation/crops_hybrid_unet_resnet34_cb55.pth
      tools/LineSegmentationEvaluator.jar       <- optional; DIVA metrics skipped without it
      20_detection/colab/                       <- receives checkpoints + results

Re-running is safe: finished arms are restored from Drive and skipped.

## 1. Config

In [16]:
# ----- edit these if your Drive layout differs -----------------------------
REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"
DRIVE_ROOT  = "/content/drive/MyDrive/Thesis"
REPO_DIR    = "/content/repo"
WORK_DIR    = "/content/work"
SUBSET      = "CS18"  # Change to CSG864 or CB55 for other subsets

# Pinned so the numbers line up with the local runs.
TRANSFORMERS_PIN = "5.14.1"

# --- the recipe, identical to the local PvtV2-b0 run -----------------------
DETR_CHECKPOINT           = "PekingU/rtdetr_r50vd"
PREFERRED_IMAGE_SIZE      = 1152
NUM_EPOCHS                = 100
PER_DEVICE_BATCH_SIZE     = 1
GRADIENT_ACCUMULATION     = 2
LEARNING_RATE             = 1e-4
WEIGHT_DECAY              = 1e-4
WARMUP_RATIO              = 0.0
MAX_GRAD_NORM             = 0.1
LR_SCHEDULER_TYPE         = "constant"
USE_EMA, EMA_DECAY        = True, 0.993
SEED                      = 42
NUM_WORKERS               = 2
COLOR_JITTER = dict(brightness=0.4, contrast=0.4, saturation=0.7, hue=0.015)

SMOKE = False        # True -> 2 epochs per arm, for a quick end-to-end check
# --------------------------------------------------------------------------

import os, shutil, subprocess, sys, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_ROOT, REPO_DIR, WORK_DIR = map(Path, (DRIVE_ROOT, REPO_DIR, WORK_DIR))
DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
# Append SUBSET to output paths to avoid overwriting previous models
DRIVE_OUT    = DRIVE_ROOT / "20_detection" / f"colab_{SUBSET}"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"
LOCAL_OUT    = WORK_DIR / "20_detection" / SUBSET

def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

print("colab runtime:", IN_COLAB)


colab runtime: True


## 2. Mount Drive

In [17]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} not found -- create it on your Drive first"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} not found -- upload 00_data there"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("drive OK:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive OK: /content/drive/MyDrive/Thesis


## 3. Clone the repo

Only the **tracked** helpers are used from the clone: `50_modelling/02_2stage/dataset.py`
and `evaluate.py` supply the PAGE-XML writer, the row de-duplication and the crop
segmenter for section 9. The training loop is defined inline in this notebook rather
than imported, because `20_detection/` is gitignored locally and is not on the remote.

In [18]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

for d in (LOCAL_DATA, LOCAL_MODELS, LOCAL_OUT):
    d.mkdir(parents=True, exist_ok=True)

# The eval helpers resolve ../../00_data etc. relative to the repo root, so recreate
# the gitignored trees there as symlinks into the local SSD dirs.
for name, target in (("00_data", LOCAL_DATA), ("80_models", LOCAL_MODELS)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(f"{link.name:12} -> {os.readlink(link)}")

STAGE_DIR = REPO_DIR / "50_modelling" / "02_2stage"
assert (STAGE_DIR / "dataset.py").exists() and (STAGE_DIR / "evaluate.py").exists(), \
    "eval helpers missing from the clone"

$ git clone --depth 1 --branch main https://github.com/Maserezio/fsl-tl-manuscripts.git /content/repo
00_data      -> /content/work/00_data
80_models    -> /content/work/80_models


## 4. Pull the CB55 data

Copies Drive -> local SSD. Training off the FUSE mount is what to avoid: ~50-100 ms per
file open starves the dataloader.

In [19]:
# Training reads COCO json + the yolo_dataset image split; evaluation additionally
# needs the raw pages, the TASK-2 PAGE ground truth and the pixel-level ground truth.
NEEDED_DIRS = [
    f"DIVA-HisDB/coco_dataset_{SUBSET}",
    f"DIVA-HisDB/yolo_dataset_{SUBSET}/images",
    f"DIVA-HisDB/{SUBSET}/img-{SUBSET}/img/validation",
    f"DIVA-HisDB/{SUBSET}/img-{SUBSET}/img/public-test",
    f"DIVA-HisDB/{SUBSET}/PAGE-gt-{SUBSET}-TASK-2/TASK-2/validation",
    f"DIVA-HisDB/{SUBSET}/PAGE-gt-{SUBSET}-TASK-2/TASK-2/public-test",
    f"DIVA-HisDB/{SUBSET}/pixel-level-gt-{SUBSET}/pixel-level-gt/public-test",
]
SEG_REL = f"02_2stage/diva-hisdb/segmentation/crops_hybrid_unet_resnet34_{SUBSET.lower()}.pth"
JAR_REL = "tools/LineSegmentationEvaluator.jar"


def _files(root: Path):
    return {p.relative_to(root): p.stat().st_size
            for p in root.rglob("*") if p.is_file() and p.suffix != ".cache"}


def sync_dir(src: Path, dst: Path, label: str):
    """Compare file lists, not directory existence -- a copy killed by a dropped
    runtime leaves a directory that exists but is half full, and an existence check
    would then skip it forever."""
    if not src.is_dir():
        print(f"[MISSING on Drive] {label}")
        return False
    want = _files(src)
    have = _files(dst) if dst.is_dir() else {}
    todo = [r for r, sz in want.items() if have.get(r) != sz]
    if not todo:
        print(f"[ok]       {label:52} {len(want):5} files")
        return True
    t0, mb = time.time(), 0.0
    for r in todo:
        d = dst / r
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src / r, d)
        mb += want[r] / 1e6
    print(f"[{'copied  ' if not have else 'repaired'}] {label:52} "
          f"{len(todo):5} files {mb:8.1f} MB in {time.time()-t0:5.1f}s")
    return True


ok = all([sync_dir(DRIVE_DATA / r, LOCAL_DATA / r, r) for r in NEEDED_DIRS])
assert ok, f"some {SUBSET} directories are missing on Drive -- see [MISSING] lines above"

seg_src, seg_dst = DRIVE_MODELS / SEG_REL, LOCAL_MODELS / SEG_REL
if seg_src.exists():
    seg_dst.parent.mkdir(parents=True, exist_ok=True)
    if not seg_dst.exists():
        shutil.copy2(seg_src, seg_dst)
    print("crop segmenter:", seg_dst)
else:
    print("[WARN] crop segmenter missing -> section 9 (DIVA metrics) will be skipped")

JAR = DRIVE_ROOT / JAR_REL
print("DIVA evaluator jar:", JAR if JAR.exists() else "MISSING -> DIVA metrics skipped")


[copied  ] DIVA-HisDB/coco_dataset_CS18                             3 files     16.4 MB in   1.5s
[copied  ] DIVA-HisDB/yolo_dataset_CS18/images                     40 files     87.4 MB in  19.8s
[copied  ] DIVA-HisDB/CS18/img-CS18/img/validation                 10 files     21.4 MB in   3.9s
[copied  ] DIVA-HisDB/CS18/img-CS18/img/public-test                10 files     21.8 MB in   3.8s
[copied  ] DIVA-HisDB/CS18/PAGE-gt-CS18-TASK-2/TASK-2/validation    20 files      0.6 MB in   8.2s
[copied  ] DIVA-HisDB/CS18/PAGE-gt-CS18-TASK-2/TASK-2/public-test    20 files      0.6 MB in   7.7s
[copied  ] DIVA-HisDB/CS18/pixel-level-gt-CS18/pixel-level-gt/public-test    10 files      3.4 MB in   4.0s
[WARN] crop segmenter missing -> section 9 (DIVA metrics) will be skipped
DIVA evaluator jar: MISSING -> DIVA metrics skipped


## 5. Dependencies

In [20]:
sh(f'pip -q install "transformers=={TRANSFORMERS_PIN}" torchmetrics '
   f'segmentation-models-pytorch albumentations pycocotools')
sh("apt-get -qq install -y default-jre > /dev/null 2>&1 || true", check=False)
sh("nvidia-smi || true", check=False)

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert transformers.__version__.startswith(TRANSFORMERS_PIN.split(".")[0]), \
    "transformers major version differs from the pin -- restart the runtime and re-run"
BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
print("bf16 supported:", BF16, "(T4 -> False, falls back to fp16)")

$ pip -q install "transformers==5.14.1" torchmetrics segmentation-models-pytorch albumentations pycocotools
$ apt-get -qq install -y default-jre > /dev/null 2>&1 || true
$ nvidia-smi || true
torch 2.11.0+cu128 | transformers 5.14.1
bf16 supported: True (T4 -> False, falls back to fp16)


## 6. Resolution guard

Measures a real forward+backward peak for **b2** and picks the largest multiple of 32
that fits with headroom. If that is below 1152, the local b0 and stock R50-vd rows are
no longer comparable, so both are added to the run list automatically.

In [21]:
import gc
from transformers import AutoConfig, RTDetrConfig, AutoModelForObjectDetection

PVT_OUT_INDICES = [1, 2, 3]   # stage2/3/4 = strides 8/16/32; [2,3,4] is invalid for PvtV2

def make_config(backbone_ckpt, num_labels=1):
    bc = AutoConfig.from_pretrained(backbone_ckpt)
    bc.out_indices = PVT_OUT_INDICES if bc.model_type == "pvt_v2" else [2, 3, 4]
    # ConvNextBackbone does `[hidden_sizes[0]] + hidden_sizes`, which raises
    # "can only concatenate list (not tuple) to list" on the tuple default.
    if isinstance(getattr(bc, "hidden_sizes", None), tuple):
        bc.hidden_sizes = list(bc.hidden_sizes)
    return RTDetrConfig.from_pretrained(
        DETR_CHECKPOINT, backbone_config=bc, use_timm_backbone=False,
        num_feature_levels=3, num_labels=num_labels,
        id2label={0: "TextLine"}, label2id={"TextLine": 0})


def peak_gib(model, res):
    torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
    lbl = [{"class_labels": torch.zeros(60, dtype=torch.long, device="cuda"),
            "boxes": torch.rand(60, 4, device="cuda").clamp(0.05, 0.9)}]
    dt = torch.bfloat16 if BF16 else torch.float16
    with torch.autocast("cuda", dtype=dt):
        out = model(pixel_values=torch.randn(1, 3, res, res, device="cuda"), labels=lbl)
    out.loss.backward()
    model.zero_grad(set_to_none=True)
    p = torch.cuda.max_memory_allocated() / 2**30
    del out; gc.collect(); torch.cuda.empty_cache()
    return p


_probe = AutoModelForObjectDetection.from_pretrained(
    DETR_CHECKPOINT, config=make_config("OpenGVLab/pvt_v2_b2"),
    ignore_mismatched_sizes=True).cuda().train()

total_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
BUDGET = total_gib * 0.80          # leave room for eval + allocator fragmentation
IMAGE_SIZE = None
for res in [r for r in (1152, 1024, 960, 896, 832, 768) if r <= PREFERRED_IMAGE_SIZE]:
    try:
        p = peak_gib(_probe, res)
        print(f"  res={res:5} peak={p:5.2f} GiB  budget={BUDGET:.2f} GiB "
              f"{'OK' if p <= BUDGET else 'too tight'}")
        if p <= BUDGET:
            IMAGE_SIZE = res
            break
    except torch.OutOfMemoryError:
        print(f"  res={res:5} OOM")
        gc.collect(); torch.cuda.empty_cache()

del _probe; gc.collect(); torch.cuda.empty_cache()
assert IMAGE_SIZE, "PvtV2-b2 does not fit even at 768 -- use a larger GPU runtime"
assert IMAGE_SIZE % 32 == 0
print(f"\nGPU {total_gib:.1f} GiB -> IMAGE_SIZE = {IMAGE_SIZE}")

ARMS = [("pvt_v2_b2_rand", "OpenGVLab/pvt_v2_b2", False),
        ("pvt_v2_b2_in1k", "OpenGVLab/pvt_v2_b2", True)]
if IMAGE_SIZE != 1152:
    print(f"[!] {IMAGE_SIZE} != 1152, so the local b0/stock rows are NOT comparable.\n"
          f"    Adding them to the run list at {IMAGE_SIZE}.")
    ARMS += [("pvt_v2_b0_rand", "OpenGVLab/pvt_v2_b0", False),
             ("stock_r50vd",    None,                  False)]
for n, ck, pre in ARMS:
    print(f"  {n:16} backbone={ck or 'stock R50-vd (COCO)'} pretrained={pre}")

[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `80`.


[transformers] RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r50vd
Key                                                                                                                   | Status     |                                                                                                        
----------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------------------------
model.backbone.model.encoder.stages.{0, 1, 2, 3}.layers.{0, 1, 2, 3, 4, 5}.layer.{0, 1, 2}.normalization.running_var  | UNEXPECTED |                                                                                                        
model.backbone.model.encoder.stages.{0, 1, 2, 3}.layers.{0, 1, 2, 3, 4, 5}.layer.{0, 1, 2}.convolution.weight         | UNEXPECTED |                                                                              

  res= 1152 peak= 7.82 GiB  budget=11.65 GiB OK

GPU 14.6 GiB -> IMAGE_SIZE = 1152
  pvt_v2_b2_rand   backbone=OpenGVLab/pvt_v2_b2 pretrained=False
  pvt_v2_b2_in1k   backbone=OpenGVLab/pvt_v2_b2 pretrained=True


## 7. Model builder

`use_pretrained_backbone=True` is a **no-op** in transformers 5.x: `load_backbone()`
always calls `AutoBackbone.from_config(...)`, which is random init — it never fetches
weights. The ImageNet arm therefore loads an `AutoBackbone` explicitly and grafts its
`state_dict` onto the detector's backbone, asserting that every key matched.

In [22]:
from transformers import AutoBackbone
from transformers.models.rt_detr.modeling_rt_detr import replace_batch_norm as _rtdetr_replace_bn


def _inner_backbone(model):
    """The AutoBackbone instance inside RT-DETR's conv encoder wrapper."""
    bb = model.model.backbone
    return bb.model if hasattr(bb, "model") else bb


def build_model(backbone_ckpt, pretrained):
    if backbone_ckpt is None:                      # stock detector, untouched
        model = AutoModelForObjectDetection.from_pretrained(
            DETR_CHECKPOINT, id2label={0: "TextLine"}, label2id={"TextLine": 0},
            ignore_mismatched_sizes=True)
    else:
        cfg = make_config(backbone_ckpt)
        # from_pretrained(config=...) keeps the COCO encoder/decoder and reinitialises
        # only the backbone, encoder_input_proj (channel counts change) and the
        # single-class heads.
        model = AutoModelForObjectDetection.from_pretrained(
            DETR_CHECKPOINT, config=cfg, ignore_mismatched_sizes=True)
        if pretrained:
            ref = AutoBackbone.from_pretrained(
                backbone_ckpt, out_indices=cfg.backbone_config.out_indices)
            missing, unexpected = _inner_backbone(model).load_state_dict(
                ref.state_dict(), strict=False)
            if missing or unexpected:
                raise RuntimeError(
                    f"backbone graft mismatch for {backbone_ckpt}: "
                    f"{len(missing)} missing, {len(unexpected)} unexpected "
                    f"(first missing: {list(missing)[:3]})")
            k = next(k for k, v in ref.state_dict().items() if v.dim() > 1)
            assert torch.allclose(_inner_backbone(model).state_dict()[k],
                                  ref.state_dict()[k]), "graft did not take"
            print(f"  grafted ImageNet weights from {backbone_ckpt}")
            del ref

    # With per-device batch 1, any live BatchNorm2d recomputes statistics from a single
    # sample every step. Swapping the module class (rather than .eval()) survives the
    # model.train() that Trainer issues each epoch. No-op for LayerNorm backbones.
    before = sum(1 for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d))
    _rtdetr_replace_bn(model)
    after = sum(1 for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d))
    n_bb = sum(p.numel() for p in model.model.backbone.parameters())
    print(f"  froze {before - after} BatchNorm2d | backbone params {n_bb:,} | "
          f"total {sum(p.numel() for p in model.parameters()):,}")
    return model, n_bb

## 8. Data pipeline, metrics, EMA, training loop

In [23]:
from collections import defaultdict
from dataclasses import dataclass
from functools import partial

import numpy as np
import torchvision.transforms
from datasets import Dataset, DatasetDict, Image as HFImage
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers import (AutoImageProcessor, Trainer, TrainerCallback,
                          TrainingArguments, set_seed)
from transformers.image_transforms import center_to_corners_format

COCO_ROOT  = LOCAL_DATA / "DIVA-HisDB/coco_dataset_CB55"
IMAGE_ROOT = LOCAL_DATA / "DIVA-HisDB/yolo_dataset_CB55/images"


def load_coco_split(split, cat2label):
    payload = json.loads((COCO_ROOT / f"{split}.json").read_text())
    by_image = defaultdict(list)
    for a in payload["annotations"]:
        by_image[a["image_id"]].append(a)
    records = []
    for info in sorted(payload["images"], key=lambda i: i["id"]):
        w, h = int(info["width"]), int(info["height"])
        path = IMAGE_ROOT / split / info["file_name"]
        if not path.is_file():
            raise FileNotFoundError(path)
        boxes, labels, areas, ids = [], [], [], []
        for a in by_image[info["id"]]:
            x, y, bw, bh = map(float, a["bbox"])
            x, y = min(max(x, 0.0), float(w)), min(max(y, 0.0), float(h))
            bw, bh = min(bw, w - x), min(bh, h - y)
            if bw <= 0 or bh <= 0:
                continue
            boxes.append([x, y, bw, bh]); labels.append(cat2label[int(a["category_id"])])
            areas.append(bw * bh); ids.append(int(a["id"]))
        if not boxes:
            raise ValueError(f"{path} has no valid boxes")
        records.append({"image_id": int(info["id"]), "image": str(path),
                        "width": w, "height": h,
                        "objects": {"id": ids, "bbox": boxes,
                                    "category": labels, "area": areas}})
    return Dataset.from_list(records).cast_column("image", HFImage())


_cats = sorted(json.loads((COCO_ROOT / "train.json").read_text())["categories"],
               key=lambda c: c["id"])
CAT2LABEL = {int(c["id"]): i for i, c in enumerate(_cats)}
ID2LABEL  = {i: c["name"] for i, c in enumerate(_cats)}
LABEL2ID  = {n: i for i, n in ID2LABEL.items()}
RAW = DatasetDict({s: load_coco_split(s, CAT2LABEL) for s in ("train", "val", "test")})
for s, d in RAW.items():
    print(f"{s:5} {len(d):3} pages, {sum(len(o['bbox']) for o in d['objects']):5} boxes")

JITTER = torchvision.transforms.ColorJitter(**COLOR_JITTER)


def transform_batch(examples, image_processor, augment):
    images, anns = [], []
    for image_id, image, objects in zip(examples["image_id"], examples["image"],
                                        examples["objects"]):
        image = image.convert("RGB")
        if augment:
            image = JITTER(image)
        images.append(np.array(image, copy=True))
        anns.append({"image_id": image_id, "annotations": [
            {"image_id": image_id, "category_id": c, "iscrowd": 0, "area": ar, "bbox": list(b)}
            for c, ar, b in zip(objects["category"], objects["area"], objects["bbox"])]})
    return image_processor(images=images, annotations=anns, return_tensors="pt")


def collate_fn(batch):
    out = {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
           "labels": [b["labels"] for b in batch]}
    if "pixel_mask" in batch[0]:
        out["pixel_mask"] = torch.stack([b["pixel_mask"] for b in batch])
    return out


@dataclass
class DetectionOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


@torch.no_grad()
def compute_metrics(evaluation_results, image_processor):
    preds, targets = evaluation_results.predictions, evaluation_results.label_ids
    sizes, proc_t, proc_p = [], [], []
    for tb in targets:
        batch = []
        for t in tb:
            v = np.atleast_1d(np.asarray(t["orig_size"])).flatten()
            size = (int(v[0]), int(v[1])); batch.append(size)
            b = center_to_corners_format(torch.as_tensor(t["boxes"]))
            b = b * torch.tensor([[size[1], size[0], size[1], size[0]]], dtype=b.dtype)
            proc_t.append({"boxes": b,
                           "labels": torch.as_tensor(t["class_labels"], dtype=torch.int64)})
        sizes.append(torch.tensor(batch))
    for pb, ts in zip(preds, sizes):
        proc_p.extend(image_processor.post_process_object_detection(
            DetectionOutput(torch.as_tensor(pb[1]), torch.as_tensor(pb[2])),
            threshold=0.0, target_sizes=ts))
    m = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=False)
    m.update(proc_p, proc_t)
    return {k: round(v.item(), 6) for k, v in m.compute().items()}


class EMACallback(TrainerCallback):
    """Shadow kept on CPU: a GPU-resident fp32 copy costs ~100-170 MB and the update
    runs only ~10x per epoch, so the transfer is irrelevant."""

    def __init__(self, decay):
        self.decay, self.shadow = decay, None

    def on_step_end(self, args, state, control, model=None, **kw):
        if model is None:
            return
        with torch.no_grad():
            msd = model.state_dict()
            if self.shadow is None:      # built here: the model is on GPU only by now
                self.shadow = {k: v.detach().float().cpu().clone()
                               for k, v in msd.items() if v.is_floating_point()}
                return
            for k, sh_ in self.shadow.items():
                sh_.mul_(self.decay).add_(msd[k].detach().float().cpu(),
                                          alpha=1.0 - self.decay)

    def copy_to(self, model):
        with torch.no_grad():
            msd = model.state_dict()
            for k, sh_ in self.shadow.items():
                msd[k].copy_(sh_.to(device=msd[k].device, dtype=msd[k].dtype))


def train_arm(name, backbone_ckpt, pretrained):
    set_seed(SEED)
    out_dir = LOCAL_OUT / name / "output"
    epochs = 2 if SMOKE else NUM_EPOCHS
    print(f"\n{'='*78}\n{name}  backbone={backbone_ckpt or 'stock R50-vd'}  "
          f"pretrained={pretrained}  imgsz={IMAGE_SIZE}  epochs={epochs}\n{'='*78}")

    image_processor = AutoImageProcessor.from_pretrained(
        DETR_CHECKPOINT, do_resize=True,
        size={"max_height": IMAGE_SIZE, "max_width": IMAGE_SIZE},
        do_pad=True, pad_size={"height": IMAGE_SIZE, "width": IMAGE_SIZE})
    ds = DatasetDict({s: d.with_transform(partial(
        transform_batch, image_processor=image_processor, augment=(s == "train")))
        for s, d in RAW.items()})

    model, n_bb = build_model(backbone_ckpt, pretrained)
    steps = max(1, len(RAW["train"]) // (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION))
    print(f"  ~{steps} optimizer steps/epoch -> ~{steps * epochs} total")

    args = TrainingArguments(
        output_dir=str(out_dir), num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO, lr_scheduler_type=LR_SCHEDULER_TYPE,
        optim="adamw_torch", max_grad_norm=MAX_GRAD_NORM,
        bf16=BF16, fp16=not BF16,
        dataloader_num_workers=NUM_WORKERS,
        eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
        metric_for_best_model="eval_map", greater_is_better=True,
        load_best_model_at_end=True, save_total_limit=1,
        remove_unused_columns=False, eval_do_concat_batches=False,
        report_to="none", seed=SEED, data_seed=SEED)

    ema = EMACallback(EMA_DECAY) if USE_EMA else None
    trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                      eval_dataset=ds["val"], processing_class=image_processor,
                      data_collator=collate_fn,
                      compute_metrics=partial(compute_metrics,
                                              image_processor=image_processor),
                      callbacks=[ema] if ema else None)
    t0 = time.time()
    trainer.train()

    raw_val = trainer.evaluate(ds["val"], metric_key_prefix="validation")
    print(f"  raw  val mAP50={raw_val['validation_map_50']:.4f} "
          f"mAP50-95={raw_val['validation_map']:.4f}")
    used, val = "raw", raw_val
    if ema is not None and ema.shadow is not None:
        keep = {k: v.detach().clone() for k, v in trainer.model.state_dict().items()}
        ema.copy_to(trainer.model)
        ema_val = trainer.evaluate(ds["val"], metric_key_prefix="validation")
        print(f"  EMA  val mAP50={ema_val['validation_map_50']:.4f} "
              f"mAP50-95={ema_val['validation_map']:.4f}")
        if ema_val["validation_map"] > raw_val["validation_map"]:
            used, val = "ema", ema_val
        else:
            trainer.model.load_state_dict(keep)
        print(f"  -> keeping {used} weights")

    test = trainer.evaluate(ds["test"], metric_key_prefix="test")
    print(f"  TEST mAP50={test['test_map_50']:.4f} mAP50-95={test['test_map']:.4f} "
          f"mAR@100={test['test_mar_100']:.4f}")

    best = out_dir / "best_model"
    trainer.save_model(str(best)); image_processor.save_pretrained(str(best))
    summary = {
        "arm": name, "backbone_ckpt": backbone_ckpt or DETR_CHECKPOINT,
        "pretrained_backbone": bool(pretrained), "image_size": IMAGE_SIZE,
        "epochs": epochs, "optimizer_steps": trainer.state.global_step,
        "weights_used": used, "params_backbone": n_bb,
        "params_total": sum(p.numel() for p in trainer.model.parameters()),
        "val_mAP50": val["validation_map_50"], "val_mAP50_95": val["validation_map"],
        "test_mAP50": test["test_map_50"], "test_mAP50_95": test["test_map"],
        "test_mAR100": test["test_mar_100"],
        "runtime_min": round((time.time() - t0) / 60, 1),
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    del trainer, model, ds
    gc.collect(); torch.cuda.empty_cache()
    return summary

train  20 pages,  1507 boxes
val    10 pages,   901 boxes
test   10 pages,   845 boxes


## 9. Train the arms

Sequential; each finished arm is pushed to Drive immediately, so a dropped runtime
costs at most one arm. Re-running restores finished arms and skips them.

In [ ]:
def pull_arm(name):
    src, dst = DRIVE_OUT / name, LOCAL_OUT / name
    if (src / "output" / "summary.json").exists() and not (dst / "output" / "summary.json").exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"[pull] {name}")


def push_arm(name):
    src, dst = LOCAL_OUT / name, DRIVE_OUT / name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"[push] {name}")


SUMMARIES = {}
for name, ck, pre in ARMS:
    pull_arm(name)
    sfile = LOCAL_OUT / name / "output" / "summary.json"
    if sfile.exists():
        SUMMARIES[name] = json.loads(sfile.read_text())
        print(f"[skip] {name} already done "
              f"(test mAP50={SUMMARIES[name]['test_mAP50']:.4f})")
        continue
    try:
        SUMMARIES[name] = train_arm(name, ck, pre)
        push_arm(name)
    except Exception as e:
        print(f"[FAILED] {name}: {type(e).__name__}: {e}")

print("\ndone:", list(SUMMARIES))


pvt_v2_b2_rand  backbone=OpenGVLab/pvt_v2_b2  pretrained=False  imgsz=1152  epochs=100


[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `80`.


[transformers] RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r50vd
Key                                                                                                                   | Status     |                                                                                                        
----------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------------------------
model.backbone.model.encoder.stages.{0, 1, 2, 3}.layers.{0, 1, 2, 3, 4, 5}.layer.{0, 1, 2}.normalization.running_var  | UNEXPECTED |                                                                                                        
model.backbone.model.encoder.stages.{0, 1, 2, 3}.layers.{0, 1, 2, 3, 4, 5}.layer.{0, 1, 2}.convolution.weight         | UNEXPECTED |                                                                              

  froze 42 BatchNorm2d | backbone params 24,849,856 | total 43,411,619
  ~10 optimizer steps/epoch -> ~1000 total


Epoch,Training Loss,Validation Loss,Map,Map 50,Map 75,Map Small,Map Medium,Map Large,Mar 1,Mar 10,Mar 100,Mar Small,Mar Medium,Mar Large,Map Per Class,Mar 100 Per Class,Classes
1,92.294788,92.793930,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0
2,93.497833,120.883972,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0
3,112.706030,120.793045,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0
4,76.773560,36.780560,0.000316,0.002274,0.000000,-1.000000,0.000000,0.000354,0.000000,0.000666,0.006215,-1.000000,0.000000,0.006236,-1.000000,-1.000000,0
5,34.842722,21.110153,0.003605,0.019052,0.000091,-1.000000,0.000000,0.003614,0.000777,0.003663,0.039068,-1.000000,0.000000,0.039198,-1.000000,-1.000000,0
6,29.722668,23.137020,0.000814,0.006143,0.000000,-1.000000,0.000000,0.000818,0.000222,0.001443,0.010322,-1.000000,0.000000,0.010356,-1.000000,-1.000000,0
7,64.631378,116.578232,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0
8,108.958594,120.690651,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0
9,99.138165,110.279991,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0
10,98.264728,24.874660,0.009723,0.044105,0.000786,-1.000000,0.000000,0.009750,0.001665,0.012209,0.065261,-1.000000,0.000000,0.065479,-1.000000,-1.000000,0


...

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


## 10. PAGE-XML + DIVA metrics

Same pipeline as the local runs, so LineIU/PixelIU are comparable: detect ->
**filter to the GT `TextRegion`** -> merge duplicate rows -> crop-segment each box into
a polygon -> PAGE-XML -> official Java evaluator.

The region filter is not optional. TASK-2 ground truth only annotates lines inside that
region, while the COCO training set also labels the marginal gloss lines outside it;
skipping the filter turns every correct gloss detection into a false positive.

`post_process_object_detection` is called with `target_sizes=(max_dim, max_dim)`, not
`(H, W)`: the processor pads to a square canvas, so boxes are normalised against the
longest side. Using `(H, W)` compresses x by `W/H` (0.75 on CB55) and drives LineIU to 0.

In [ ]:
import xml.etree.ElementTree as ET
from xml.dom import minidom
import cv2, pandas as pd

sys.path.insert(0, str(STAGE_DIR))
from dataset import create_page_xml, mask_to_polygon, polygon_to_baseline
from evaluate import _remove_overlapping_rows, load_segm_model, segment_crop

TEST_IMG = LOCAL_DATA / "DIVA-HisDB/CB55/img-CB55/img/public-test"
VAL_IMG  = LOCAL_DATA / "DIVA-HisDB/CB55/img-CB55/img/validation"
GT_PAGE  = LOCAL_DATA / "DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2/public-test"
GT_VAL   = LOCAL_DATA / "DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2/validation"
GT_PIXEL = LOCAL_DATA / "DIVA-HisDB/CB55/pixel-level-gt-CB55/pixel-level-gt/public-test"
NS = "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"
RESIZE_W, RESIZE_H, BIN_THRESH = 1088, 128, 0.1
THRESHOLD_GRID = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
DIVA_OK = JAR.exists() and seg_dst.exists()


def region_poly(xml_path):
    el = ET.parse(xml_path).getroot().find(f".//{{{NS}}}TextRegion/{{{NS}}}Coords")
    return None if el is None else np.array(
        [list(map(int, xy.split(","))) for xy in el.attrib["points"].split()],
        dtype=np.int32).reshape(-1, 2)


def detect(model, proc, img_path, thr):
    from PIL import Image
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    inputs = proc(images=im, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**inputs)
    r = proc.post_process_object_detection(out, threshold=thr,
                                           target_sizes=[(max(W, H), max(W, H))])[0]
    b = r["boxes"].cpu().numpy().reshape(-1, 4)
    if len(b):
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, W); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H)
        b = b[(b[:, 2] > b[:, 0]) & (b[:, 3] > b[:, 1])]
    return b, W, H


def keep_in_region(boxes, xml_path):
    if not len(boxes) or not Path(xml_path).exists():
        return boxes
    poly = region_poly(xml_path)
    if poly is None:
        return boxes
    m = [cv2.pointPolygonTest(poly, (float((b[0]+b[2])/2), float((b[1]+b[3])/2)), False) >= 0
         for b in boxes]
    return boxes[np.array(m, dtype=bool)]


def diva_eval(name):
    """Returns dict of mean DIVA metrics, or None if prerequisites are missing."""
    if not DIVA_OK:
        return None
    mdl_dir = LOCAL_OUT / name / "output" / "best_model"
    if not mdl_dir.exists():
        return None
    model = AutoModelForObjectDetection.from_pretrained(mdl_dir).eval().cuda()
    proc = AutoImageProcessor.from_pretrained(mdl_dir)
    seg = load_segm_model(str(seg_dst), "resnet34", "unet", "cuda")

    # threshold chosen on VAL, never on test
    best_thr, best_err = THRESHOLD_GRID[0], None
    for thr in THRESHOLD_GRID:
        err = 0
        for f in sorted(VAL_IMG.glob("*.jpg")):
            gx = GT_VAL / f"{f.stem}.xml"
            if not gx.exists():
                continue
            b, _, _ = detect(model, proc, f, thr)
            b = _remove_overlapping_rows(keep_in_region(b, gx))
            err += abs(len(b) - len(ET.parse(gx).getroot().findall(f".//{{{NS}}}TextLine")))
        if best_err is None or err < best_err:
            best_thr, best_err = thr, err
    print(f"  [{name}] conf={best_thr} (val |diff|={best_err})")

    xml_dir = LOCAL_OUT / name / "pred_xml"
    if xml_dir.exists():
        shutil.rmtree(xml_dir)
    xml_dir.mkdir(parents=True)
    for f in sorted(TEST_IMG.glob("*.jpg")):
        img = cv2.imread(str(f))
        b, W, H = detect(model, proc, f, best_thr)
        b = _remove_overlapping_rows(keep_in_region(b, GT_PAGE / f"{f.stem}.xml"))
        root, region = create_page_xml(str(f), W, H)
        for i, box in enumerate(b):
            x1, y1, x2, y2 = (int(v) for v in box)
            x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, W), min(y2, H)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            mask = segment_crop(crop, seg, RESIZE_W, RESIZE_H, "cuda", BIN_THRESH)
            fg = float((mask > 0).mean())
            if fg > 0.95 or fg < 0.005:
                continue
            poly = mask_to_polygon(mask, x1, y1)
            if poly is None or len(poly) < 3:
                continue
            base = polygon_to_baseline(poly)
            if len(base) < 2:
                continue
            tl = ET.SubElement(region, "TextLine", {"id": f"textline_{i}", "custom": "0"})
            ET.SubElement(tl, "Coords", {"points": " ".join(f"{x},{y}" for x, y in poly)})
            ET.SubElement(tl, "Baseline", {"points": " ".join(f"{x},{y}" for x, y in base)})
            ET.SubElement(ET.SubElement(tl, "TextEquiv"), "Unicode").text = ""
        (xml_dir / f"{f.stem}.xml").write_text(
            minidom.parseString(ET.tostring(root, encoding="utf-8")).toprettyxml(indent="  "))

    agg = xml_dir / "results.csv"      # the tool APPENDS here; stdout is just logging
    if agg.exists():
        agg.unlink()
    for x in sorted(xml_dir.glob("*.xml")):
        subprocess.run(["java", "-cp", str(JAR), "ch.unifr.LineSegmentationEvaluatorTool",
                        "-igt", str(GT_PIXEL / f"{x.stem}.png"),
                        "-xgt", str(GT_PAGE / f"{x.stem}.xml"),
                        "-xp", str(x), "-overlap", str(TEST_IMG / f"{x.stem}.jpg"), "-csv"],
                       cwd=str(xml_dir), capture_output=True, text=True)
    del model, seg
    gc.collect(); torch.cuda.empty_cache()
    if not agg.exists():
        print(f"  [{name}] evaluator produced no results.csv")
        return None
    d = pd.read_csv(agg)
    return {"conf": best_thr, "LineIU": d["LinesIU"].mean(),
            "PixelIU": d["PixelIU"].mean(), "LineF1": d["LinesFMeasure"].mean(),
            "LineP": d["LinesPrecision"].mean(), "LineR": d["LinesRecall"].mean()}


DIVA = {}
if not DIVA_OK:
    print("skipping DIVA metrics: "
          + ("evaluator jar missing " if not JAR.exists() else "")
          + ("crop segmenter missing" if not seg_dst.exists() else ""))
else:
    for name in SUMMARIES:
        try:
            r = diva_eval(name)
            if r:
                DIVA[name] = r
                push_arm(name)
                print(f"  [{name}] LineIU={r['LineIU']:.4f} PixelIU={r['PixelIU']:.4f}")
        except Exception as e:
            print(f"  [{name}] DIVA eval failed: {type(e).__name__}: {e}")

## 11. Summary

Colab arms alongside the local rows. Rows measured at a different `imgsz` are marked —
compare only within one resolution.

In [ ]:
import pandas as pd

# Local reference rows (this repo, 8GB RTX 4060, imgsz=1152, same recipe).
LOCAL_ROWS = [
    dict(run="stock R50-vd (COCO)",   where="local", imgsz=1152, params_backbone=23_474_016,
         test_mAP50=0.8820, test_mAP50_95=0.7733, test_mAR100=0.8208,
         LineIU=0.9589, PixelIU=0.8819, LineF1=0.9788),
    dict(run="ConvNeXt-tiny (random)", where="local", imgsz=1152, params_backbone=27_821_280,
         test_mAP50=0.8177, test_mAP50_95=0.6132, test_mAR100=0.7043,
         LineIU=0.9135, PixelIU=0.8345, LineF1=0.9534),
    dict(run="PvtV2-b0 (random)",      where="local", imgsz=1152, params_backbone=3_409_760,
         test_mAP50=0.8739, test_mAP50_95=0.6528, test_mAR100=0.7183,
         LineIU=0.9563, PixelIU=0.8739, LineF1=0.9772),
]

LABEL = {"pvt_v2_b2_rand": "PvtV2-b2 (random)",
         "pvt_v2_b2_in1k": "PvtV2-b2 (ImageNet)",
         "pvt_v2_b0_rand": "PvtV2-b0 (random)",
         "stock_r50vd":    "stock R50-vd (COCO)"}

rows = list(LOCAL_ROWS)
for name, s in SUMMARIES.items():
    d = DIVA.get(name, {})
    rows.append(dict(run=LABEL.get(name, name), where="colab", imgsz=s["image_size"],
                     params_backbone=s["params_backbone"],
                     test_mAP50=round(s["test_mAP50"], 4),
                     test_mAP50_95=round(s["test_mAP50_95"], 4),
                     test_mAR100=round(s["test_mAR100"], 4),
                     LineIU=round(d["LineIU"], 4) if d else None,
                     PixelIU=round(d["PixelIU"], 4) if d else None,
                     LineF1=round(d["LineF1"], 4) if d else None))

df = pd.DataFrame(rows)
df["params_backbone"] = df["params_backbone"].map("{:,}".format)
print(df.to_string(index=False))

if len(set(r["imgsz"] for r in rows)) > 1:
    print("\n[!] mixed imgsz in this table -- rows are only comparable within one "
          "resolution. Section 6 re-ran the local baselines when it had to lower it.")

out_csv = LOCAL_OUT / "backbone_matrix.csv"
df.to_csv(out_csv, index=False)
shutil.copy2(out_csv, DRIVE_OUT / "backbone_matrix.csv")
print("\nsaved ->", DRIVE_OUT / "backbone_matrix.csv")